# Critique Experiment 3: Combined Scenario (Shock Train + Higher Bad-Regime Uncertainty)

Combined stress test:
- deterministic switch to bad regime,
- sequential cost-push pulses,
- higher uncertainty in bad regime.

No retraining, only counterfactual simulations.

In [ ]:
import os
import sys
import pathlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


def _find_project_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir():
        return cand
    raise RuntimeError("Could not find project root containing src/.")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ModelParams, TrainConfig
from src.io_utils import load_json
from src.table2_builder import _load_run_dir, _load_net_from_run
from src.steady_states import solve_flexprice_sss, solve_efficient_sss
from src.sss_from_policy import switching_policy_sss_by_regime_from_policy
from src.deqn import Trainer, implied_nominal_rate_from_euler, _transition_probs_to_next
from src.model_common import unpack_state, shock_laws_of_motion, identities
from src.metrics import output_gap_from_consumption
from src.sanity_checks import _state_from_policy_sss

ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts" / "runs")
COMPUTE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
POLICIES = ["taylor", "discretion", "commitment"]

np.set_printoptions(suppress=True, precision=4)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)
print("COMPUTE_DEVICE:", COMPUTE_DEVICE)


def _parse_dtype(v):
    s = str(v)
    if "float64" in s:
        return torch.float64
    if "float32" in s:
        return torch.float32
    if "bfloat16" in s:
        return torch.bfloat16
    return torch.float32


def load_params_from_run(run_dir: str, *, device: str | None = None) -> ModelParams:
    cfg = load_json(os.path.join(run_dir, "config.json"))
    p = cfg.get("params", {})
    keep = {k: v for k, v in p.items() if k in ModelParams.__dataclass_fields__}
    keep["dtype"] = _parse_dtype(p.get("dtype", "float32"))
    keep["device"] = device or COMPUTE_DEVICE
    return ModelParams(**keep).to_torch()


def maybe_rbar(params: ModelParams, policy: str):
    if policy not in ("mod_taylor", "mod_taylor_zlb"):
        return None
    flex = solve_flexprice_sss(params)
    return torch.tensor(
        [flex.by_regime[0]["r_star"], flex.by_regime[1]["r_star"]],
        device=params.device,
        dtype=params.dtype,
    )


def load_policy_bundle(policy: str):
    run_dir = _load_run_dir(ARTIFACTS_ROOT, policy, use_selected=True, required_files=())
    params = load_params_from_run(run_dir, device=COMPUTE_DEVICE)
    net = _load_net_from_run(run_dir, params, policy)
    rbar = maybe_rbar(params, policy)
    sss = switching_policy_sss_by_regime_from_policy(
        params,
        net,
        policy=policy,
        rbar_by_regime=rbar,
    ).by_regime
    commit_dim = None
    if policy == "commitment":
        try:
            commit_dim = int(net.net[0].in_features)
        except Exception:
            commit_dim = 8
    x0 = _state_from_policy_sss(
        params,
        policy,
        sss[0],
        regime=0,
        commitment_state_dim=commit_dim,
    )
    return {
        "policy": policy,
        "run_dir": run_dir,
        "params": params,
        "net": net,
        "rbar": rbar,
        "sss": sss,
        "x0": x0,
    }


def _build_next_state(policy: str, x_cur: torch.Tensor, st, out, logA_n, logg_n, xi_n, s_n, params: ModelParams):
    dt = params.dtype
    if policy in ("taylor", "mod_taylor", "taylor_zlb", "mod_taylor_zlb", "discretion", "discretion_zlb"):
        return torch.stack([out["Delta"], logA_n, logg_n, xi_n, s_n.to(dt)], dim=-1)

    if policy == "taylor_para":
        if int(x_cur.shape[-1]) >= 7:
            if st.p21 is not None:
                p21_prev = st.p21
            else:
                p21_prev = torch.full_like(out["Delta"], float(params.p21))
            return torch.stack([out["Delta"], logA_n, logg_n, xi_n, s_n.to(dt), out["i_nom"], p21_prev], dim=-1)
        return torch.stack([out["Delta"], logA_n, logg_n, xi_n, s_n.to(dt)], dim=-1)

    if policy == "commitment":
        if int(x_cur.shape[-1]) >= 8:
            return torch.stack([out["Delta"], logA_n, logg_n, xi_n, s_n.to(dt), out["vartheta"], out["varrho"], out["c"]], dim=-1)
        return torch.stack([out["Delta"], logA_n, logg_n, xi_n, s_n.to(dt), out["vartheta"], out["varrho"]], dim=-1)

    if policy == "commitment_zlb":
        return torch.stack(
            [
                out["Delta"],
                logA_n,
                logg_n,
                xi_n,
                s_n.to(dt),
                out["vartheta"],
                out["varrho"],
                out["c"],
                out["i_nom"],
                out["varphi"],
            ],
            dim=-1,
        )

    raise ValueError(f"Unsupported policy: {policy}")


@torch.inference_mode()
def simulate_custom(
    params: ModelParams,
    policy: str,
    net,
    x0: torch.Tensor,
    *,
    T: int,
    regime_path=None,
    epsA_path=None,
    epsg_path=None,
    epst_path=None,
    noise_scale: float = 0.0,
    bad_sigma_mult: float = 1.0,
    seed: int = 123,
    rbar_by_regime=None,
    gh_n: int = 3,
):
    torch.manual_seed(int(seed))
    np.random.seed(int(seed))

    if epsA_path is None:
        epsA_path = np.zeros(T, dtype=np.float64)
    if epsg_path is None:
        epsg_path = np.zeros(T, dtype=np.float64)
    if epst_path is None:
        epst_path = np.zeros(T, dtype=np.float64)

    epsA_path = np.asarray(epsA_path, dtype=np.float64).reshape(-1)
    epsg_path = np.asarray(epsg_path, dtype=np.float64).reshape(-1)
    epst_path = np.asarray(epst_path, dtype=np.float64).reshape(-1)
    if not (len(epsA_path) == len(epsg_path) == len(epst_path) == int(T)):
        raise ValueError("eps paths must all have length T")

    if regime_path is not None:
        regime_path = [int(v) for v in regime_path]
        if len(regime_path) != int(T) + 1:
            raise ValueError("regime_path must have length T+1")

    if params.device == "cpu":
        cfg_sim = TrainConfig.dev(seed=0, cpu_num_threads=None, cpu_num_interop_threads=None)
    else:
        cfg_sim = TrainConfig.full(seed=0, cpu_num_threads=None, cpu_num_interop_threads=None)

    tr = Trainer(params=params, cfg=cfg_sim, policy=policy, net=net, gh_n=int(gh_n), rbar_by_regime=rbar_by_regime)

    x = x0.to(device=params.device, dtype=params.dtype)
    B = int(x.shape[0])

    store = {
        "c": np.zeros((T + 1, B), dtype=np.float64),
        "pi": np.zeros((T + 1, B), dtype=np.float64),
        "i": np.zeros((T + 1, B), dtype=np.float64),
        "Delta": np.zeros((T + 1, B), dtype=np.float64),
        "y": np.zeros((T + 1, B), dtype=np.float64),
        "h": np.zeros((T + 1, B), dtype=np.float64),
        "g": np.zeros((T + 1, B), dtype=np.float64),
        "A": np.zeros((T + 1, B), dtype=np.float64),
        "tau": np.zeros((T + 1, B), dtype=np.float64),
        "s": np.zeros((T + 1, B), dtype=np.int64),
        "logA": np.zeros((T + 1, B), dtype=np.float64),
        "loggtilde": np.zeros((T + 1, B), dtype=np.float64),
        "xi": np.zeros((T + 1, B), dtype=np.float64),
    }

    explicit_i_policies = ("taylor", "taylor_para", "mod_taylor", "taylor_zlb", "mod_taylor_zlb", "commitment_zlb")

    for t in range(T + 1):
        out = tr._policy_outputs(x)
        st = unpack_state(x, policy)
        ids = identities(params, st, out)

        if policy in explicit_i_policies:
            i_t = out["i_nom"]
        else:
            i_t = implied_nominal_rate_from_euler(params, policy, x, out, int(gh_n), tr)

        store["c"][t] = out["c"].detach().cpu().numpy()
        store["pi"][t] = out["pi"].detach().cpu().numpy()
        store["i"][t] = i_t.detach().cpu().numpy()
        store["Delta"][t] = out["Delta"].detach().cpu().numpy()
        store["y"][t] = ids["y"].detach().cpu().numpy()
        store["h"][t] = ids["h"].detach().cpu().numpy()
        store["g"][t] = ids["g"].detach().cpu().numpy()
        store["A"][t] = ids["A"].detach().cpu().numpy()
        store["tau"][t] = (ids["one_plus_tau"] - 1.0).detach().cpu().numpy()
        store["s"][t] = st.s.detach().cpu().numpy()
        store["logA"][t] = st.logA.detach().cpu().numpy()
        store["loggtilde"][t] = st.loggtilde.detach().cpu().numpy()
        store["xi"][t] = st.xi.detach().cpu().numpy()

        if t == T:
            break

        epsA = torch.full((B,), float(epsA_path[t]), device=params.device, dtype=params.dtype)
        epsg = torch.full((B,), float(epsg_path[t]), device=params.device, dtype=params.dtype)
        epst = torch.full((B,), float(epst_path[t]), device=params.device, dtype=params.dtype)

        if float(noise_scale) != 0.0:
            z = torch.randn((B,), device=params.device, dtype=params.dtype)
            mult = torch.where(
                st.s == int(params.bad_state),
                torch.full((B,), float(bad_sigma_mult), device=params.device, dtype=params.dtype),
                torch.ones((B,), device=params.device, dtype=params.dtype),
            )
            epst = epst + float(noise_scale) * z * mult

        if regime_path is not None:
            s_next = torch.full((B,), int(regime_path[t + 1]), device=params.device, dtype=torch.long)
        else:
            u = torch.rand((B,), device=params.device, dtype=params.dtype)
            p0, _ = _transition_probs_to_next(params, st)
            s_next = torch.where(u < p0, torch.zeros_like(st.s), torch.ones_like(st.s))

        logA_n, logg_n, xi_n, s_n = shock_laws_of_motion(params, st, epsA, epsg, epst, s_next)
        x = _build_next_state(policy, x, st, out, logA_n, logg_n, xi_n, s_n, params)

    return store


def add_output_gap(sim: dict, params: ModelParams) -> dict:
    eff = solve_efficient_sss(params)
    shp = np.asarray(sim["c"]).shape
    x = output_gap_from_consumption(sim, eff, params=params, time_varying=True)
    sim["x"] = np.asarray(x, dtype=np.float64).reshape(shp)
    return sim


ann = lambda x: 400.0 * np.asarray(x)


In [ ]:
T = 80
B = 512
regime_path = [0] + [1] * T
shock_times = [0, 4, 8, 12]
shock_size = 1.25
bad_sigma_mult = 2.0

base_epst = np.zeros(T, dtype=np.float64)
comb_epst = np.zeros(T, dtype=np.float64)
for k in shock_times:
    if 0 <= k < T:
        comb_epst[k] = shock_size

print("T=", T, "B=", B)
print("shock_times=", shock_times, "shock_size=", shock_size)
print("bad_sigma_mult=", bad_sigma_mult)


In [ ]:
def qband(a, q=(0.1, 0.5, 0.9)):
    a = np.asarray(a, dtype=np.float64)
    return tuple(np.quantile(a, qq, axis=1) for qq in q)


rows = []

for policy in POLICIES:
    try:
        bundle = load_policy_bundle(policy)
    except Exception as e:
        print(f"[skip] policy={policy}: {e}")
        continue

    params = bundle["params"]
    net = bundle["net"]
    rbar = bundle["rbar"]
    x0 = bundle["x0"].repeat(B, 1)

    sim_base = simulate_custom(
        params,
        policy,
        net,
        x0,
        T=T,
        regime_path=regime_path,
        epst_path=base_epst,
        noise_scale=1.0,
        bad_sigma_mult=1.0,
        seed=123,
        rbar_by_regime=rbar,
    )
    sim_comb = simulate_custom(
        params,
        policy,
        net,
        x0,
        T=T,
        regime_path=regime_path,
        epst_path=comb_epst,
        noise_scale=1.0,
        bad_sigma_mult=bad_sigma_mult,
        seed=123,
        rbar_by_regime=rbar,
    )

    sim_base = add_output_gap(sim_base, params)
    sim_comb = add_output_gap(sim_comb, params)

    t = np.arange(T + 1)
    pi_b_q10, pi_b_q50, pi_b_q90 = qband(ann(sim_base["pi"]))
    pi_c_q10, pi_c_q50, pi_c_q90 = qband(ann(sim_comb["pi"]))

    x_b_q10, x_b_q50, x_b_q90 = qband(100.0 * sim_base["x"])
    x_c_q10, x_c_q50, x_c_q90 = qband(100.0 * sim_comb["x"])

    fig, ax = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

    ax[0].plot(t, pi_b_q50, label="baseline median", color="tab:blue")
    ax[0].fill_between(t, pi_b_q10, pi_b_q90, color="tab:blue", alpha=0.2, label="baseline 10-90")
    ax[0].plot(t, pi_c_q50, label="combined median", color="tab:red")
    ax[0].fill_between(t, pi_c_q10, pi_c_q90, color="tab:red", alpha=0.2, label="combined 10-90")
    ax[0].set_title(f"{policy}: inflation (ann. %)")
    ax[0].legend(ncol=2)

    ax[1].plot(t, x_b_q50, label="baseline median", color="tab:blue")
    ax[1].fill_between(t, x_b_q10, x_b_q90, color="tab:blue", alpha=0.2, label="baseline 10-90")
    ax[1].plot(t, x_c_q50, label="combined median", color="tab:red")
    ax[1].fill_between(t, x_c_q10, x_c_q90, color="tab:red", alpha=0.2, label="combined 10-90")
    ax[1].set_title(f"{policy}: output gap (%)")
    ax[1].set_xlabel("quarter")
    ax[1].legend(ncol=2)

    plt.tight_layout()
    plt.show()

    rows.append({
        "policy": policy,
        "median_peak_pi_base_ann_pct": float(np.max(pi_b_q50[1:])),
        "median_peak_pi_comb_ann_pct": float(np.max(pi_c_q50[1:])),
        "median_min_x_base_pct": float(np.min(x_b_q50[1:])),
        "median_min_x_comb_pct": float(np.min(x_c_q50[1:])),
    })

combined_summary = pd.DataFrame(rows)
combined_summary


In [ ]:
if not combined_summary.empty:
    out = combined_summary.copy()
    out["d_peak_pi_ann_pct"] = out["median_peak_pi_comb_ann_pct"] - out["median_peak_pi_base_ann_pct"]
    out["d_min_x_pct"] = out["median_min_x_comb_pct"] - out["median_min_x_base_pct"]
    out[["policy", "d_peak_pi_ann_pct", "d_min_x_pct"]]
